# Phase 09.2 — Overfitting Reduction Experiments (V2)

Run All is resumable. Phase 09 artifacts remain untouched. The notebook uses source train/validation rows only and never loads test rows. Primary models: Swin-Tiny and ConvNeXt-Tiny; VGG16 is a low-fit comparison. Four settings: control, regularization, augmentation, combined.


In [1]:
# CELL 1 — Imports, configuration, data/model helpers
from pathlib import Path
import os, json, time, random, gc, warnings
import numpy as np, pandas as pd
from PIL import Image, ImageFile
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
SPLITS = ROOT/"data"/"splits_v2"
OUT = ROOT/"data"/"results"/"phase09_2_overfitting_v2"
OUT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED, BS, MAX_EPOCHS, PATIENCE = 42, 32, 100, 12
MODELS = ["swin_tiny", "convnext_tiny", "vgg16"]
DATASETS = ["cric", "riva"]
EXPS = {
 "control": {"aug":"none", "wd":1e-5, "smooth":0.0},
 "regularization": {"aug":"none", "wd":1e-4, "smooth":0.05},
 "augmentation": {"aug":"moderate", "wd":1e-5, "smooth":0.0},
 "combined": {"aug":"moderate", "wd":1e-4, "smooth":0.05},
}
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)

def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def split_file(ds):
    p = SPLITS/f"{ds}_splits_v2.csv"
    if not p.exists(): raise FileNotFoundError(f"Missing split CSV: {p}")
    return p

def load_df(ds):
    d = pd.read_csv(split_file(ds)); cols = {c.lower():c for c in d.columns}
    def pick(*names): return next((cols[n] for n in names if n in cols), None)
    pc, lc, sc = pick("crop_path","image_path","filepath","path"), pick("binary_label","label","target"), pick("split","set","subset")
    if not all([pc,lc,sc]): raise ValueError(f"{ds}: expected path/label/split columns; found {list(d.columns)}")
    d = d.copy(); d["_path"] = d[pc].astype(str); d["_label"] = pd.to_numeric(d[lc]).astype(int)
    d["_split"] = d[sc].astype(str).str.lower().replace({"training":"train","validation":"val","valid":"val","dev":"val","testing":"test"})
    d = d[d["_split"].isin(["train","val"])].copy()  # Test rows are deliberately excluded.
    d["_path"] = d["_path"].apply(lambda p: str((ROOT/p).resolve()) if not Path(p).is_absolute() else p)
    if not {"train","val"}.issubset(set(d["_split"])): raise ValueError(f"{ds}: train/val split missing")
    if not set(d["_label"].unique()).issubset({0,1}): raise ValueError(f"{ds}: labels must be 0/1")
    return d

def make_transforms(model, aug):
    size = 224 if model == "vit_b16" else 128
    a = []
    if aug == "moderate":
        a = [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
             transforms.RandomRotation(12),
             transforms.ColorJitter(brightness=.08, contrast=.08, saturation=.05, hue=.015)]
    norm = [transforms.Resize((size,size)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)]
    return transforms.Compose(a+norm), transforms.Compose(norm)

class Crops(Dataset):
    def __init__(self, df, tf): self.df, self.tf = df.reset_index(drop=True), tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; p = Path(r["_path"])
        if not p.exists(): raise FileNotFoundError(f"Image not found: {p}")
        with Image.open(p) as im: x = self.tf(im.convert("RGB"))
        return x, int(r["_label"])

def build_model(name):
    if name == "swin_tiny":
        m = models.swin_t(weights=models.Swin_T_Weights.IMAGENET1K_V1); m.head = nn.Linear(m.head.in_features,2)
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1); m.classifier[2] = nn.Linear(m.classifier[2].in_features,2)
    elif name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1); m.classifier[6] = nn.Linear(m.classifier[6].in_features,2)
    else: raise ValueError(name)
    return m

def metrics(y, prob):
    y, prob = np.asarray(y,dtype=int), np.asarray(prob)
    pred = (prob >= .5).astype(int); tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    try: auc = float(roc_auc_score(y,prob))
    except Exception: auc = float("nan")
    return {"accuracy":float(accuracy_score(y,pred)), "precision":float(precision_score(y,pred,zero_division=0)),
      "recall_sensitivity":float(recall_score(y,pred,zero_division=0)),
      "specificity":float(tn/(tn+fp)) if tn+fp else float("nan"),
      "macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)), "auroc":auc,
      "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys=[]; probs=[]; loss_sum=0.; n=0; ce=nn.CrossEntropyLoss()
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); z=model(x); loss_sum += ce(z,y).item()*len(y); n += len(y)
        ys += y.cpu().tolist(); probs += torch.softmax(z,1)[:,1].cpu().tolist()
    m=metrics(ys,probs); m["loss"]=loss_sum/max(n,1); return m

def train_epoch(model, loader, opt, loss_fn):
    model.train(); total=0.; n=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(set_to_none=True)
        loss=loss_fn(model(x),y); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        total += loss.item()*len(y); n += len(y)
    return total/max(n,1)

def atomic_save(obj, p):
    tmp=Path(str(p)+".tmp"); torch.save(obj,tmp); os.replace(tmp,p)

print("Project:",ROOT,"| Device:",DEVICE)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
print("Planned runs:",len(MODELS)*len(DATASETS)*len(EXPS))


Project: c:\Users\nanda\Documents\cervical-semi-sl | Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Planned runs: 24


In [2]:
# CELL 2 — Resumable experiment runner; checkpoints and history saved every epoch
def run_one(exp, model_name, ds, cfg):
    seed_all(); rd=OUT/exp/model_name/ds; rd.mkdir(parents=True,exist_ok=True)
    latest,bestp,hp,sp = rd/"checkpoint_latest.pt",rd/"checkpoint_best.pt",rd/"history.csv",rd/"status.json"
    if sp.exists() and bestp.exists() and hp.exists() and json.loads(sp.read_text()).get("status")=="completed":
        print("SKIP completed:",exp,model_name,ds); return
    d=load_df(ds); tr=d[d["_split"]=="train"]; va=d[d["_split"]=="val"]
    ttf,evf=make_transforms(model_name,cfg["aug"])
    trl=DataLoader(Crops(tr,ttf),batch_size=BS,shuffle=True,num_workers=0,pin_memory=torch.cuda.is_available())
    tre=DataLoader(Crops(tr,evf),batch_size=BS,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    val=DataLoader(Crops(va,evf),batch_size=BS,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())
    model=build_model(model_name).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=cfg["wd"])
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode="max",factor=.5,patience=4)
    lossfn=nn.CrossEntropyLoss(label_smoothing=cfg["smooth"])
    config={"experiment":exp,"model":model_name,"dataset":ds,"seed":SEED,"batch_size":BS,"max_epochs":MAX_EPOCHS,
      "patience":PATIENCE,"lr":1e-4,"weight_decay":cfg["wd"],"label_smoothing":cfg["smooth"],
      "augmentation":cfg["aug"],"selection":"source validation Macro-F1","test_evaluated":False,"split_file":str(split_file(ds))}
    (rd/"config.json").write_text(json.dumps(config,indent=2))
    hist=pd.read_csv(hp).to_dict("records") if hp.exists() else []
    start,best,best_epoch,stale=0,-1.,0,0
    if latest.exists():
        c=torch.load(latest,map_location=DEVICE,weights_only=False)
        model.load_state_dict(c["model"]); opt.load_state_dict(c["optimizer"]); sched.load_state_dict(c["scheduler"])
        start,best,best_epoch,stale=c["epoch"],c["best"],c["best_epoch"],c["stale"]
        print("RESUME",exp,model_name,ds,"from epoch",start+1)
    sp.write_text(json.dumps({"status":"running","last_epoch":start},indent=2))
    for ep in range(start,MAX_EPOCHS):
        t0=time.time(); train_epoch(model,trl,opt,lossfn); tm=evaluate(model,tre); vm=evaluate(model,val); sched.step(vm["macro_f1"])
        row={"epoch":ep+1,"train_loss":tm["loss"],"train_macro_f1":tm["macro_f1"],"train_accuracy":tm["accuracy"],
          "val_loss":vm["loss"],"val_macro_f1":vm["macro_f1"],"val_accuracy":vm["accuracy"],
          "val_precision":vm["precision"],"val_recall_sensitivity":vm["recall_sensitivity"],
          "val_specificity":vm["specificity"],"val_auroc":vm["auroc"],"lr":opt.param_groups[0]["lr"],"seconds":time.time()-t0}
        hist=[r for r in hist if int(r["epoch"])!=ep+1]+[row]; pd.DataFrame(hist).sort_values("epoch").to_csv(hp,index=False)
        if vm["macro_f1"] > best:
            best,best_epoch,stale=vm["macro_f1"],ep+1,0
            atomic_save({"model":model.state_dict(),"epoch":ep+1,"best_val_macro_f1":best,"config":config},bestp)
            (rd/"best_validation_metrics.json").write_text(json.dumps(vm,indent=2))
        else: stale += 1
        atomic_save({"model":model.state_dict(),"optimizer":opt.state_dict(),"scheduler":sched.state_dict(),
          "epoch":ep+1,"best":best,"best_epoch":best_epoch,"stale":stale},latest)
        sp.write_text(json.dumps({"status":"running","last_epoch":ep+1,"best_epoch":best_epoch,"best_val_macro_f1":best,"stale":stale},indent=2))
        print(f"{exp:14} {model_name:13} {ds.upper():4} ep {ep+1:03}/{MAX_EPOCHS} trainF1={tm['macro_f1']:.4f} valF1={vm['macro_f1']:.4f} best={best:.4f}@{best_epoch} stale={stale}/{PATIENCE}")
        if stale >= PATIENCE: break
    sp.write_text(json.dumps({"status":"completed","best_epoch":best_epoch,"best_val_macro_f1":best,
      "last_epoch":int(pd.read_csv(hp).epoch.max())},indent=2))
    del model,opt,sched,trl,tre,val; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("DONE:",rd)

# CELL 3 — Run All; completed runs skip and interrupted runs resume
for exp,cfg in EXPS.items():
    for model_name in MODELS:
        for ds in DATASETS:
            run_one(exp,model_name,ds,cfg)
print("All configured runs processed.")


SKIP completed: control swin_tiny cric
SKIP completed: control swin_tiny riva
SKIP completed: control convnext_tiny cric
SKIP completed: control convnext_tiny riva
SKIP completed: control vgg16 cric
SKIP completed: control vgg16 riva
SKIP completed: regularization swin_tiny cric
SKIP completed: regularization swin_tiny riva
SKIP completed: regularization convnext_tiny cric
SKIP completed: regularization convnext_tiny riva
SKIP completed: regularization vgg16 cric
SKIP completed: regularization vgg16 riva
SKIP completed: augmentation swin_tiny cric
SKIP completed: augmentation swin_tiny riva
SKIP completed: augmentation convnext_tiny cric
SKIP completed: augmentation convnext_tiny riva
SKIP completed: augmentation vgg16 cric
SKIP completed: augmentation vgg16 riva
SKIP completed: combined swin_tiny cric
SKIP completed: combined swin_tiny riva
SKIP completed: combined convnext_tiny cric
SKIP completed: combined convnext_tiny riva
RESUME combined vgg16 cric from epoch 33
combined       vg

In [3]:
# CELL 4 — Consolidated validation-only results; test set is never read
rows=[]
for exp in EXPS:
 for m in MODELS:
  for ds in DATASETS:
   rd=OUT/exp/m/ds; hp=rd/"history.csv"
   if not hp.exists(): continue
   h=pd.read_csv(hp)
   if h.empty: continue
   r=h.loc[h["val_macro_f1"].idxmax()]
   st=json.loads((rd/"status.json").read_text()) if (rd/"status.json").exists() else {}
   rows.append({"experiment":exp,"model":m,"dataset":ds,"status":st.get("status","unknown"),
    "best_epoch":int(r["epoch"]),"train_macro_f1":float(r["train_macro_f1"]),
    "val_macro_f1":float(r["val_macro_f1"]),"train_val_gap":float(r["train_macro_f1"]-r["val_macro_f1"]),
    "val_loss":float(r["val_loss"]),"val_accuracy":float(r["val_accuracy"]),"val_precision":float(r["val_precision"]),
    "val_recall_sensitivity":float(r["val_recall_sensitivity"]),"val_specificity":float(r["val_specificity"]),
    "val_auroc":float(r["val_auroc"]),"run_dir":str(rd)})
summary=pd.DataFrame(rows)
if not summary.empty:
 summary.to_csv(OUT/"phase09_2_overfitting_summary.csv",index=False)
 controls=summary[summary.experiment=="control"][["model","dataset","val_macro_f1","train_val_gap"]].rename(
  columns={"val_macro_f1":"control_val_f1","train_val_gap":"control_gap"})
 cmp=summary.merge(controls,on=["model","dataset"],how="left")
 cmp["delta_val_macro_f1"]=cmp.val_macro_f1-cmp.control_val_f1
 cmp["gap_reduction"]=cmp.control_gap-cmp.train_val_gap
 cmp["gap_reduction_pct"]=np.where(cmp.control_gap.abs()>1e-12,100*cmp.gap_reduction/cmp.control_gap.abs(),np.nan)
 cmp.to_csv(OUT/"phase09_2_vs_control.csv",index=False)
 display(cmp.sort_values(["dataset","model","experiment"]))
 print("Saved summaries in",OUT)
else: print("No histories found yet.")


,experiment,model,dataset,status,best_epoch,train_macro_f1,val_macro_f1,train_val_gap,val_loss,val_accuracy,val_precision,val_recall_sensitivity,val_specificity,val_auroc,run_dir,control_val_f1,control_gap,delta_val_macro_f1,gap_reduction,gap_reduction_pct
14,augmentation,convnext_tiny,cric,completed,34,0.999624,0.911386,0.088238,0.738417,0.915004,0.891540,0.895425,0.927954,0.971302,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.896728,0.103272,0.014657,0.015033,14.557204
20,combined,convnext_tiny,cric,completed,11,0.995489,0.902485,0.093004,0.311431,0.906331,0.877419,0.888889,0.917867,0.951972,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.896728,0.103272,0.005756,0.010268,9.942503
2,control,convnext_tiny,cric,completed,6,1.000000,0.896728,0.103272,0.726645,0.902862,0.920097,0.827887,0.952450,0.965079,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.896728,0.103272,0.000000,0.000000,0.000000
8,regularization,convnext_tiny,cric,completed,5,0.996993,0.897859,0.099133,0.345953,0.901995,0.874459,0.880174,0.916427,0.940114,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.896728,0.103272,0.001131,0.004139,4.007507
12,augmentation,swin_tiny,cric,completed,6,0.981835,0.896126,0.085708,0.331923,0.901127,0.891156,0.856209,0.930836,0.964313,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.902697,0.097303,-0.006571,0.011595,11.915982
18,combined,swin_tiny,cric,completed,5,0.964311,0.897285,0.067026,0.281220,0.901127,0.866242,0.888889,0.909222,0.956123,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.902697,0.097303,-0.005412,0.030277,31.116566
0,control,swin_tiny,cric,completed,39,1.000000,0.902697,0.097303,1.202453,0.907199,0.894619,0.869281,0.932277,0.955355,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.902697,0.097303,0.000000,0.000000,0.000000
6,regularization,swin_tiny,cric,completed,20,1.000000,0.896993,0.103007,0.364694,0.901127,0.872570,0.880174,0.914986,0.957036,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.902697,0.097303,-0.005704,-0.005704,-5.862166
16,augmentation,vgg16,cric,completed,17,0.987842,0.893768,0.094074,0.475101,0.898526,0.880000,0.862745,0.922190,0.961673,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.880296,0.105399,0.013472,0.011325,10.745158
22,combined,vgg16,cric,completed,25,0.997869,0.890832,0.107036,0.332838,0.895924,0.880899,0.854031,0.923631,0.946108,c:\Users\nanda\Documents\cervical-semi-sl\data...,0.880296,0.105399,0.010536,-0.001637,-1.553124


Saved summaries in c:\Users\nanda\Documents\cervical-semi-sl\data\results\phase09_2_overfitting_v2


## Decision rules

- Compare each intervention with the same model/dataset control.
- Positive `gap_reduction` means the train–validation gap shrank; positive `delta_val_macro_f1` means validation Macro-F1 improved.
- Do not choose by gap alone: prioritize validation Macro-F1, then gap/loss and abnormal recall.
- Test data is excluded; cross-dataset evaluation happens after configuration selection.
- Interrupt anytime and click Run All again. Completed runs are skipped; unfinished runs resume from the latest checkpoint.
- If you change hyperparameters, create a new key in `EXPS`; do not reuse a checkpoint directory with different settings.
